In [ ]:
import bacco
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [ ]:
import os
os.chdir("/lscratch/fgmaion/MTNG-resims/src")
import utils

In [ ]:
%load_ext autoreload
%autoreload 2

## Load the MTNG-DM at 2160^3 resolution

In [ ]:
basedir = "/cosmos_storage/simulations/MTNG/DM-Gadget4/MTNG-L500-2160-A/"

resolution_level = 1

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME
numpart = int(2160**3)

mtng_dm = bacco.Simulation(basedir=basedir, halo_file="groups_265/fof_subhalo_tab_265", dm_file="snapdir_265/snapshot_265", sim_format='gadget4_hdf5', fixedPk=True, sigma8=sigma8,\
    tau=tau, ns=ns, numpart=numpart, use_orphans=False, use_ids=False)

Load the halo selection, which is relative to the 2160 simulation

In [ ]:
dm_sel = []
with open("/cosmos_storage/data_sharing/MN5_resims/level2/dm_halo_sel_1pmbin.txt", "r") as f:
    for line in f.readlines():
        dm_sel.append(int(line))

dm_sel = np.array(dm_sel)

In [ ]:
halo_pos = mtng_dm.fof['halo_pos'][dm_sel]

In [ ]:
pos_halo = mtng_dm.get_halo_particles(ihalo=dm_sel[400], relative=False)

## Load the zooms

In [ ]:
#base = "/cosmos_storage/data_sharing/tmp_resims/1080-A/level4/haloes_10/hydro_output/"
base = "/cosmos_storage/data_sharing/MN5_resims/1pmbin_2160_legacy/1pmbin_2160_legacy/hydro_output"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

sim = bacco.Simulation(basedir=base, halo_file="groups_009/fof_subhalo_tab_009", dm_file="snapdir_009/snapshot_009", sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=False)

In [ ]:
L_p = 8
z_slab = 10
xmin = np.array([halo_pos[ihalo][0]-L_p/2,halo_pos[ihalo][1]-L_p/2,halo_pos[ihalo][2]-z_slab/2])

zoom_sel = np.where( (sim.dm['pos'][:,0]>(halo_pos[ihalo][0]-L_p/2)) & (sim.dm['pos'][:,0]<(halo_pos[ihalo][0]+L_p/2)) &\
                     (sim.dm['pos'][:,1]>(halo_pos[ihalo][1]-L_p/2)) & (sim.dm['pos'][:,1]<(halo_pos[ihalo][1]+L_p/2)) &\
                     (sim.dm['pos'][:,2]>(halo_pos[ihalo][2]-z_slab/2)) & (sim.dm['pos'][:,2]<(halo_pos[ihalo][2]+z_slab/2)) )[0]

zoom_grid = bacco.statistics.compute_mesh(pos=sim.dm['pos'][zoom_sel] - xmin[np.newaxis,:], box=L_p, ngrid=512)

In [ ]:
mtng_sel = np.where( (mtng_dm.dm['pos'][:,0]>(halo_pos[ihalo][0]-L_p/2)) & (mtng_dm.dm['pos'][:,0]<(halo_pos[ihalo][0]+L_p/2)) &\
                     (mtng_dm.dm['pos'][:,1]>(halo_pos[ihalo][1]-L_p/2)) & (mtng_dm.dm['pos'][:,1]<(halo_pos[ihalo][1]+L_p/2)) &\
                     (mtng_dm.dm['pos'][:,2]>(halo_pos[ihalo][2]-z_slab/2)) & (mtng_dm.dm['pos'][:,2]<(halo_pos[ihalo][2]+z_slab/2)) )[0]

mtng_grid = bacco.statistics.compute_mesh(pos=mtng_dm.dm['pos'][mtng_sel] - xmin[np.newaxis,:], box=L_p, ngrid=512)

In [ ]:
halo_grid = bacco.statistics.compute_mesh(pos=pos_halo - xmin[np.newaxis,:], box=L_p, ngrid=512)

In [ ]:
fig, ax = plt.subplots(1,2, dpi=150, figsize=(11,5))

#ax[0].imshow(np.log10(1+np.sum(mtng_grid[0,:,:,:],axis=2)))
ax[0].imshow(np.log10(1+np.sum(mtng_grid[0,:,:,:],axis=2)))
ax[1].imshow(np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=2)))

# for i in range(halo_sel.shape):
#     circle = plt.Circle((mtng_halo[halo_sel][i,1],mtng_halo[halo_sel][i,0]), mtng_dm.fof['halo_r200c'][mtng_sel][i]/L_p*512, color='r', fill=False)
#     ax[1].add_artist(circle)

In [ ]:
pos = sim.hy['dm']['pos']
zoom_grid = bacco.statistics.compute_mesh(pos=pos - np.array([25,380,440])[np.newaxis   , :], box=30, ngrid=512)

In [ ]:
halo_pos

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5), dpi=150)

ax[0].imshow( np.log10(1+np.sum(grid_ics1[0,:,:,:],axis=0)) )
ax[0].plot(halo_pos[:,2], halo_pos[:,1], marker='*', color='y', ls='')
ax[0].set_xlabel('$z$')
ax[0].set_ylabel('$y$')

ax[1].imshow( np.log10(1+np.sum(grid_ics1[0,:,:,:],axis=1)) )
ax[1].plot(halo_pos[:,2], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[1].set_xlabel('$z$')
ax[1].set_ylabel('$x$')

ax[2].imshow( np.log10(1+np.sum(grid_ics1[0,:,:,:],axis=2)) )
ax[2].plot(halo_pos[:,1], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[2].set_xlabel('$y$')
ax[2].set_ylabel('$x$')


In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5), dpi=150)

ax[0].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=0)) )
ax[0].plot(halo_pos[:,2], halo_pos[:,1], marker='*', color='y', ls='')
ax[0].set_xlabel('$z$')
ax[0].set_ylabel('$y$')

ax[1].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=1)) )
ax[1].plot(halo_pos[:,2], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[1].set_xlabel('$z$')
ax[1].set_ylabel('$x$')

ax[2].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=2)) )
ax[2].plot(halo_pos[:,1], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[2].set_xlabel('$y$')
ax[2].set_ylabel('$x$')


In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5), dpi=150)

ax[0].set_ylim(380,410)
ax[0].set_xlim(440,470)

ax[1].set_ylim(0,50)
ax[1].set_xlim(440,470)

ax[2].set_ylim(0,50)
ax[2].set_xlim(380,410)

ax[0].imshow( np.log10(1+zoom_grid[0,30,:,:]) )
ax[0].plot(halo_pos[:,2], halo_pos[:,1], marker='*', color='y', ls='')
ax[0].set_xlabel('$z$')
ax[0].set_ylabel('$y$')

ax[1].imshow( np.log10(1+zoom_grid[0,:,395,:]) )
ax[1].plot(halo_pos[:,2], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[1].set_xlabel('$z$')
ax[1].set_ylabel('$x$')

ax[2].imshow( np.log10(1+zoom_grid[0,:,:,458]) )
ax[2].plot(halo_pos[:,1], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[2].set_xlabel('$y$')
ax[2].set_ylabel('$x$')

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5), dpi=150)

ax[0].set_ylim(380,410)
ax[0].set_xlim(440,470)

ax[1].set_ylim(0,50)
ax[1].set_xlim(440,470)

ax[2].set_ylim(0,50)
ax[2].set_xlim(380,410)

ax[0].imshow( np.log10(1+zoom_grid[0,30,:,:]) )
ax[0].plot(halo_pos[:,2], halo_pos[:,1], marker='*', color='y', ls='')
ax[0].set_xlabel('$z$')
ax[0].set_ylabel('$y$')

ax[1].imshow( np.log10(1+zoom_grid[0,:,395,:]) )
ax[1].plot(halo_pos[:,2], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[1].set_xlabel('$z$')
ax[1].set_ylabel('$x$')

ax[2].imshow( np.log10(1+zoom_grid[0,:,:,458]) )
ax[2].plot(halo_pos[:,1], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[2].set_xlabel('$y$')
ax[2].set_ylabel('$x$')


In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5), dpi=150)

ax[0].set_ylim(380,410)
ax[0].set_xlim(440,470)

ax[1].set_ylim(0,50)
ax[1].set_xlim(440,470)

ax[2].set_ylim(0,50)
ax[2].set_xlim(380,410)

ax[0].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=0)) )
ax[0].plot(halo_pos[:,2], halo_pos[:,1], marker='*', color='y', ls='')
ax[0].set_xlabel('$z$')
ax[0].set_ylabel('$y$')

ax[1].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=1)) )
ax[1].plot(halo_pos[:,2], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[1].set_xlabel('$z$')
ax[1].set_ylabel('$x$')

ax[2].imshow( np.log10(1+np.sum(zoom_grid[0,:,:,:],axis=2)) )
ax[2].plot(halo_pos[:,1], (halo_pos[:,0]-125)%500, marker='*', color='y', ls='')
ax[2].set_xlabel('$y$')
ax[2].set_ylabel('$x$')


### Let's see what are the properties of these halos

In [ ]:
m200b = 1e10*sim.fof['halo_m200b']
mfof = 1e10*sim.fof['halo_mfof_type'][:,1]

group_len = sim.fof['halo_len']

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5.5,5))

ax.hist(np.log10(m200b[np.where(m200b>0)]), bins=20, log=True, histtype='step')
ax2 = ax.twinx()
ax2.plot(np.log10(m200b[np.where(m200b>0)]), group_len[np.where(m200b>0)], marker='o', ls='', ms=0.5)
ax2.set_ylabel('Length')
ax2.set_yscale('log')

ax.set_xlabel('$M_{200,b}[M_\odot/h]$')
ax.set_ylabel('Counts')

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5.5,5))

ax.hist(np.log10(mfof[np.where(m200b>0)]), bins=20, log=True, histtype='step')
ax2 = ax.twinx()
ax2.plot(np.log10(mfof[np.where(m200b>0)]), group_len[np.where(m200b>0)], marker='o', ls='', ms=0.5)
ax2.set_ylabel('Length')
ax2.set_yscale('log')

ax.set_xlabel('$M_{fof}[M_\odot/h]$')
ax.set_ylabel('Counts')

#### Cross-match selected halos with zoomed ones

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5,5))

ax.plot(pos_sel[:,0],pos_sel[:,1], ms=0.005, ls='', marker='o')


In [ ]:
fig, ax = plt.subplots(1, 2, dpi=150, figsize=(11, 5))

def select(pos, zmin, zmax):
    sel = np.where((pos[:,2]<zmax) & (pos[:,2]>zmin))[0]
    return pos[sel,:]

gas_sel = select(sim.hy['gas']['pos'], 0, 10)
star_sel = select(sim.hy['star']['pos'], 0, 10)
bh_sel = select(sim.hy['bh']['pos'], 0, 10)
dm_sel = select(sim.hy['dm']['pos'], 0, 10)

ax[0].set_xlim(dm_sel[:,0].min(), dm_sel[:,0].max())
ax[0].set_ylim(dm_sel[:,1].min(), dm_sel[:,1].max())

ax[0].plot(dm_sel[:,0], dm_sel[:,1], marker='.', ls='', ms=0.1, color='gray')
ax[0].plot(gas_sel[:,0], gas_sel[:,1], marker='.', ls='', ms=0.05, color='r')
ax[0].plot(star_sel[:,0], star_sel[:,1], marker='*', ls='', ms=0.8, color='y')
ax[0].plot(bh_sel[:,0], bh_sel[:,1], marker='o', ls='', ms=0.3, color='k')

pos = mtng_dm.dm['pos']
pos[:,0] -= 125
pos = pos % 500

pos_sel = select(pos, 0, 10)

ax[1].set_xlim(dm_sel[:,0].min(), dm_sel[:,0].max())
ax[1].set_ylim(dm_sel[:,1].min(), dm_sel[:,1].max())

ax[1].plot(pos_sel[:,0], pos_sel[:,1], marker='.', ls='', ms=1, color='gray')



In [ ]:
fig, ax = plt.subplots(1, 2, dpi=150, figsize=(11,5))

def select(pos, zmin, zmax):
    sel = np.where((pos[:,2]<zmax) & (pos[:,2]>zmin))[0]
    return pos[sel,:]

gas_sel = select(sim.hy['gas']['pos'], 450, 455)
star_sel = select(sim.hy['star']['pos'], 450, 455)
bh_sel = select(sim.hy['bh']['pos'], 450, 455)
dm_sel = select(sim.hy['dm']['pos'], 450, 455)

ax[0].set_xlim(dm_sel[:,0].min(), dm_sel[:,0].max())
ax[0].set_ylim(dm_sel[:,1].min(), dm_sel[:,1].max())

ax[0].plot(dm_sel[:,0], dm_sel[:,1], marker='.', ls='', ms=0.1, color='gray')
# ax[0].plot(gas_sel[:,0], gas_sel[:,1], marker='.', ls='', ms=0.05, color='r')
# ax[0].plot(star_sel[:,0], star_sel[:,1], marker='*', ls='', ms=0.8, color='y')
# ax[0].plot(bh_sel[:,0], bh_sel[:,1], marker='o', ls='', ms=0.3, color='k')

pos_sel = select(pos, 450, 455)

ax[1].set_xlim(dm_sel[:,0].min(), dm_sel[:,0].max())
ax[1].set_ylim(dm_sel[:,1].min(), dm_sel[:,1].max())

ax[1].plot(pos_sel[:,0], pos_sel[:,1], marker='.', ls='', ms=0.1, color='gray')


